# 🧹 SEC EDGAR — Data Cleaning & Validation
**Project:** SEC EDGAR Financial Ratio Analysis  
**Engineer:** Meet Saini  
**Last Updated:** 2026-05-30

---

### Notebook Objectives
1. Clean and cast column data types
2. Handle empty strings and nulls in value column
3. Filter segment-level rows from edgar_num_all
4. Validate cleaned tables against master tables
5. Produce two clean tables ready for SQL analysis

### Input Tables
- `edgar_sub_all` → 192,059 rows
- `edgar_num_all` → 88,170,247 rows

### Output Tables
- `edgar_sub_clean` → cleaned filing metadata
- `edgar_num_clean` → cleaned financial numbers

## Step 1 — Environment Setup

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when, trim, to_date

spark = SparkSession.builder.getOrCreate()
print(f"✅ Spark session ready | Version: {spark.version}")

## Step 2 — Pre-Cleaning Row Counts
Baseline counts before cleaning — used for validation after cleaning.

In [0]:
sub_count = spark.table("edgar_sub_all").count()
num_count = spark.table("edgar_num_all").count()

print(f"{'='*45}")
print(f"PRE-CLEANING COUNTS")
print(f"{'='*45}")
print(f"edgar_sub_all : {sub_count:,} rows")
print(f"edgar_num_all : {num_count:,} rows")
print(f"{'='*45}")

## Step 3 — Clean `edgar_sub_all`

Cleaning steps:
- Cast `filed` from string to date
- Cast `sic` from string to integer
- Trim whitespace from `name` and `countryba`
- Save as `edgar_sub_clean`

In [0]:
sub_df = spark.table("edgar_sub_all")

sub_clean = sub_df \
    .withColumn("filed", to_date(col("filed"), "yyyyMMdd")) \
    .withColumn("sic", 
        when(trim(col("sic")) == "", None)
        .otherwise(col("sic").cast("integer"))
    ) \
    .withColumn("name", trim(col("name"))) \
    .withColumn("countryba", trim(col("countryba")))

sub_clean.write.format("delta").mode("overwrite").saveAsTable("edgar_sub_clean")

print(f"✅ edgar_sub_clean saved | Rows: {sub_clean.count():,}")

## Step 4 — Clean `edgar_num_all`

Cleaning steps:
- Filter out segment-level rows (keep segments IS NULL only)
- Replace empty strings in `value` with NULL
- Cast `value` from string to DOUBLE using TRY_CAST
- Cast `ddate` from string to date
- Remove rows where value is NULL after casting
- Save as `edgar_num_clean`

In [0]:
num_df = spark.table("edgar_num_all")

num_clean = num_df \
    .filter(trim(col("segments")) == "") \
    .withColumn("value",
        when(trim(col("value")) == "", None)
        .otherwise(col("value"))
    ) \
    .withColumn("value", col("value").cast("double")) \
    .withColumn("ddate", to_date(col("ddate"), "yyyyMMdd")) \
    .filter(col("value").isNotNull())

num_clean.write.format("delta").mode("overwrite").saveAsTable("edgar_num_clean")

print(f"✅ edgar_num_clean saved | Rows: {num_clean.count():,}")

## Step 5 — Post-Cleaning Validation

Validates:
- Row counts before and after cleaning
- No nulls in critical columns
- Data types are correctly cast
- Value column has no empty strings

In [0]:
sub_clean_count = spark.table("edgar_sub_clean").count()
num_clean_count = spark.table("edgar_num_clean").count()

print(f"{'='*55}")
print(f"POST-CLEANING VALIDATION")
print(f"{'='*55}")
print(f"{'Table':<25} {'Before':>12} {'After':>12} {'Dropped':>12}")
print(f"{'-'*55}")
print(f"{'edgar_sub_all':<25} {sub_count:>12,} {sub_clean_count:>12,} {sub_count - sub_clean_count:>12,}")
print(f"{'edgar_num_all':<25} {num_count:>12,} {num_clean_count:>12,} {num_count - num_clean_count:>12,}")
print(f"{'='*55}")

## Step 6 — Schema Validation
Confirms data types are correctly cast after cleaning.

In [0]:
print("=== edgar_sub_clean SCHEMA ===")
spark.table("edgar_sub_clean").printSchema()

print("\n=== edgar_num_clean SCHEMA ===")
spark.table("edgar_num_clean").printSchema()

## Step 7 — Null Check on Cleaned Tables
Confirms no critical nulls remain after cleaning.

In [0]:
from pyspark.sql.functions import sum as spark_sum, when

def count_nulls(table_name):
    df = spark.table(table_name)
    
    null_exprs = [] 
    for c in df.columns:
        expr = spark_sum(when(col(c).isNull(), 1).otherwise(0)).alias(c)
        null_exprs.append(expr)
    
    null_counts = df.select(null_exprs).collect()[0].asDict()
    
    print(f"\nNULL CHECK — {table_name}")
    print("-" * 40)
    for col_name, null_count in sorted(null_counts.items(), key=lambda x: -x[1]):
        if null_count > 0:
            print(f"⚠️  {col_name}: {null_count:,} nulls")
    print("✅ Done")

count_nulls("edgar_sub_clean")
count_nulls("edgar_num_clean")

## Step 8 — Sample Preview of Cleaned Tables
Final sanity check on cleaned data.

In [0]:
print("=== edgar_sub_clean PREVIEW ===")
spark.sql("""
    SELECT adsh, cik, name, sic, filed, period, source_year
    FROM edgar_sub_clean
    LIMIT 5
""").show(truncate=False)

print("=== edgar_num_clean PREVIEW ===")
spark.sql("""
    SELECT adsh, tag, ddate, qtrs, uom, value, period, source_year
    FROM edgar_num_clean
    LIMIT 5
""").show(truncate=False)

## Step 9 — Pivot Financial Data to Wide Format
Converts long format `edgar_num_clean` into wide format.
One row per filing with all ratio inputs as columns.
> ⏱️ Estimated time: 5-10 minutes

In [0]:
spark.sql("""
    CREATE OR REPLACE TABLE edgar_ratios AS
    SELECT
        adsh,
        source_year,
        source_quarter,
        period,
        MAX(CASE WHEN tag = 'Assets' THEN value END) AS total_assets,
        MAX(CASE WHEN tag = 'Liabilities' THEN value END) AS total_liabilities,
        MAX(CASE WHEN tag = 'StockholdersEquity' THEN value END) AS stockholders_equity,
        MAX(CASE WHEN tag = 'NetIncomeLoss' THEN value END) AS net_income,
        MAX(CASE WHEN tag = 'Revenues' THEN value END) AS revenues,
        MAX(CASE WHEN tag = 'RevenueFromContractWithCustomerExcludingAssessedTax' THEN value END) AS revenues_alt,
        MAX(CASE WHEN tag = 'CurrentAssets' THEN value END) AS current_assets,
        MAX(CASE WHEN tag = 'AssetsCurrent' THEN value END) AS current_assets_alt,
        MAX(CASE WHEN tag = 'CurrentLiabilities' THEN value END) AS current_liabilities,
        MAX(CASE WHEN tag = 'LiabilitiesCurrent' THEN value END) AS current_liabilities_alt,
        MAX(CASE WHEN tag = 'LongTermDebt' THEN value END) AS long_term_debt,
        MAX(CASE WHEN tag = 'OperatingIncomeLoss' THEN value END) AS operating_income,
        MAX(CASE WHEN tag = 'GrossProfit' THEN value END) AS gross_profit,
        MAX(CASE WHEN tag = 'CashAndCashEquivalentsAtCarryingValue' THEN value END) AS cash
    FROM edgar_num_clean
    WHERE qtrs IN ('4', '0')
    GROUP BY adsh, source_year, source_quarter, period
""")

count = spark.sql("SELECT COUNT(*) AS rows FROM edgar_ratios").collect()[0]["rows"]
print(f"✅ edgar_ratios created | Rows: {count:,}")

## Step 10 — Compute Individual Ratio Tables
One table per ratio for clean modular analysis.
Each table contains company metadata + the computed ratio.

### Step 10a — Current Ratio
**Formula:** Current Assets / Current Liabilities  
**Measures:** Short-term liquidity

In [0]:
spark.sql("""
    CREATE OR REPLACE TABLE ratio_current AS
    SELECT
        r.adsh, s.cik, s.name, s.sic, s.filed,
        r.source_year, r.source_quarter, r.period,
        ROUND(
            COALESCE(r.current_assets, r.current_assets_alt) /
            COALESCE(r.current_liabilities, r.current_liabilities_alt), 4
        ) AS current_ratio
    FROM edgar_ratios r
    JOIN edgar_sub_clean s ON r.adsh = s.adsh
    WHERE COALESCE(r.current_assets, r.current_assets_alt) IS NOT NULL
    AND COALESCE(r.current_liabilities, r.current_liabilities_alt) IS NOT NULL
    AND COALESCE(r.current_liabilities, r.current_liabilities_alt) > 0
""")
print(f"✅ ratio_current created | Rows: {spark.sql('SELECT COUNT(*) FROM ratio_current').collect()[0][0]:,}")

### Step 10b — Debt to Equity
**Formula:** Long Term Debt / Stockholders Equity  
**Measures:** Financial leverage

In [0]:
spark.sql("""
    CREATE OR REPLACE TABLE ratio_debt_to_equity AS
    SELECT
        r.adsh, s.cik, s.name, s.sic, s.filed,
        r.source_year, r.source_quarter, r.period,
        ROUND(r.long_term_debt / r.stockholders_equity, 4) AS debt_to_equity
    FROM edgar_ratios r
    JOIN edgar_sub_clean s ON r.adsh = s.adsh
    WHERE r.long_term_debt IS NOT NULL
    AND r.stockholders_equity IS NOT NULL
    AND r.stockholders_equity != 0
""")
print(f"✅ ratio_debt_to_equity created | Rows: {spark.sql('SELECT COUNT(*) FROM ratio_debt_to_equity').collect()[0][0]:,}")

### Step 10c — Return on Assets (ROA)
**Formula:** Net Income / Total Assets  
**Measures:** Asset efficiency

In [0]:
spark.sql("""
    CREATE OR REPLACE TABLE ratio_roa AS
    SELECT
        r.adsh, s.cik, s.name, s.sic, s.filed,
        r.source_year, r.source_quarter, r.period,
        ROUND(r.net_income / r.total_assets, 4) AS roa
    FROM edgar_ratios r
    JOIN edgar_sub_clean s ON r.adsh = s.adsh
    WHERE r.net_income IS NOT NULL
    AND r.total_assets IS NOT NULL
    AND r.total_assets > 0
""")
print(f"✅ ratio_roa created | Rows: {spark.sql('SELECT COUNT(*) FROM ratio_roa').collect()[0][0]:,}")

### Step 10d — Return on Equity (ROE)
**Formula:** Net Income / Stockholders Equity  
**Measures:** Equity efficiency

In [0]:
spark.sql("""
    CREATE OR REPLACE TABLE ratio_roe AS
    SELECT
        r.adsh, s.cik, s.name, s.sic, s.filed,
        r.source_year, r.source_quarter, r.period,
        ROUND(r.net_income / r.stockholders_equity, 4) AS roe
    FROM edgar_ratios r
    JOIN edgar_sub_clean s ON r.adsh = s.adsh
    WHERE r.net_income IS NOT NULL
    AND r.stockholders_equity IS NOT NULL
    AND r.stockholders_equity != 0
""")
print(f"✅ ratio_roe created | Rows: {spark.sql('SELECT COUNT(*) FROM ratio_roe').collect()[0][0]:,}")

### Step 10e — Gross Margin
**Formula:** Gross Profit / Revenue  
**Measures:** Production efficiency

In [0]:
spark.sql("""
    CREATE OR REPLACE TABLE ratio_gross_margin AS
    SELECT
        r.adsh, s.cik, s.name, s.sic, s.filed,
        r.source_year, r.source_quarter, r.period,
        ROUND(r.gross_profit / COALESCE(r.revenues, r.revenues_alt), 4) AS gross_margin
    FROM edgar_ratios r
    JOIN edgar_sub_clean s ON r.adsh = s.adsh
    WHERE r.gross_profit IS NOT NULL
    AND COALESCE(r.revenues, r.revenues_alt) IS NOT NULL
    AND COALESCE(r.revenues, r.revenues_alt) > 0
""")
print(f"✅ ratio_gross_margin created | Rows: {spark.sql('SELECT COUNT(*) FROM ratio_gross_margin').collect()[0][0]:,}")

### Step 10f — Operating Margin
**Formula:** Operating Income / Revenue  
**Measures:** Operational efficiency

In [0]:
spark.sql("""
    CREATE OR REPLACE TABLE ratio_operating_margin AS
    SELECT
        r.adsh, s.cik, s.name, s.sic, s.filed,
        r.source_year, r.source_quarter, r.period,
        ROUND(r.operating_income / COALESCE(r.revenues, r.revenues_alt), 4) AS operating_margin
    FROM edgar_ratios r
    JOIN edgar_sub_clean s ON r.adsh = s.adsh
    WHERE r.operating_income IS NOT NULL
    AND COALESCE(r.revenues, r.revenues_alt) IS NOT NULL
    AND COALESCE(r.revenues, r.revenues_alt) > 0
""")
print(f"✅ ratio_operating_margin created | Rows: {spark.sql('SELECT COUNT(*) FROM ratio_operating_margin').collect()[0][0]:,}")

### Step 10g — Debt Ratio
**Formula:** Total Liabilities / Total Assets  
**Measures:** Overall leverage

In [0]:
spark.sql("""
    CREATE OR REPLACE TABLE ratio_debt_ratio AS
    SELECT
        r.adsh, s.cik, s.name, s.sic, s.filed,
        r.source_year, r.source_quarter, r.period,
        ROUND(r.total_liabilities / r.total_assets, 4) AS debt_ratio
    FROM edgar_ratios r
    JOIN edgar_sub_clean s ON r.adsh = s.adsh
    WHERE r.total_liabilities IS NOT NULL
    AND r.total_assets IS NOT NULL
    AND r.total_assets > 0
""")
print(f"✅ ratio_debt_ratio created | Rows: {spark.sql('SELECT COUNT(*) FROM ratio_debt_ratio').collect()[0][0]:,}")

### Step 10h — Asset Turnover
**Formula:** Revenue / Total Assets  
**Measures:** Asset utilization

In [0]:
spark.sql("""
    CREATE OR REPLACE TABLE ratio_asset_turnover AS
    SELECT
        r.adsh, s.cik, s.name, s.sic, s.filed,
        r.source_year, r.source_quarter, r.period,
        ROUND(COALESCE(r.revenues, r.revenues_alt) / r.total_assets, 4) AS asset_turnover
    FROM edgar_ratios r
    JOIN edgar_sub_clean s ON r.adsh = s.adsh
    WHERE COALESCE(r.revenues, r.revenues_alt) IS NOT NULL
    AND r.total_assets IS NOT NULL
    AND r.total_assets > 0
""")
print(f"✅ ratio_asset_turnover created | Rows: {spark.sql('SELECT COUNT(*) FROM ratio_asset_turnover').collect()[0][0]:,}")

## Step 11 — Verify All Ratio Tables
Confirm all 8 ratio tables created successfully with row counts.

In [0]:
ratio_tables = [
    "ratio_current",
    "ratio_debt_to_equity",
    "ratio_roa",
    "ratio_roe",
    "ratio_gross_margin",
    "ratio_operating_margin",
    "ratio_debt_ratio",
    "ratio_asset_turnover"
]

print(f"{'='*45}")
print(f"RATIO TABLE SUMMARY")
print(f"{'='*45}")
print(f"{'Table':<30} {'Rows':>12}")
print(f"{'-'*45}")
for table in ratio_tables:
    count = spark.sql(f"SELECT COUNT(*) FROM {table}").collect()[0][0]
    print(f"{table:<30} {count:>12,}")
print(f"{'='*45}")

## Step 12 — Build Unified KPI Table
Joins all 8 ratio tables into one unified table.
All downstream SQL analysis queries run against this single table.
> ⏱️ Estimated time: 2-3 minutes

In [0]:
spark.sql("""
    CREATE OR REPLACE TABLE edgar_kpi_unified AS
    SELECT
        cr.adsh,
        cr.cik,
        cr.name,
        cr.sic,
        cr.filed,
        cr.source_year,
        cr.source_quarter,
        cr.period,
        cr.current_ratio,
        dr.debt_ratio,
        roa.roa,
        roe.roe,
        om.operating_margin,
        gm.gross_margin,
        at.asset_turnover,
        dte.debt_to_equity
    FROM ratio_current cr
    LEFT JOIN ratio_debt_ratio dr ON cr.adsh = dr.adsh
    LEFT JOIN ratio_roa roa ON cr.adsh = roa.adsh
    LEFT JOIN ratio_roe roe ON cr.adsh = roe.adsh
    LEFT JOIN ratio_operating_margin om ON cr.adsh = om.adsh
    LEFT JOIN ratio_gross_margin gm ON cr.adsh = gm.adsh
    LEFT JOIN ratio_asset_turnover at ON cr.adsh = at.adsh
    LEFT JOIN ratio_debt_to_equity dte ON cr.adsh = dte.adsh
""")

count = spark.sql("SELECT COUNT(*) FROM edgar_kpi_unified").collect()[0][0]
print(f"✅ edgar_kpi_unified created | Rows: {count:,}")

In [0]:
spark.sql("""
    CREATE OR REPLACE TABLE edgar_kpi_clean AS
    SELECT *
    FROM edgar_kpi_unified
    WHERE (current_ratio IS NULL OR current_ratio BETWEEN 0 AND 50)
    AND (debt_ratio IS NULL OR debt_ratio BETWEEN 0 AND 10)
    AND (roa IS NULL OR roa BETWEEN -5 AND 5)
    AND (roe IS NULL OR roe BETWEEN -5 AND 5)
    AND (operating_margin IS NULL OR operating_margin BETWEEN -5 AND 5)
    AND (gross_margin IS NULL OR gross_margin BETWEEN -2 AND 2)
    AND (asset_turnover IS NULL OR asset_turnover BETWEEN 0 AND 10)
    AND (debt_to_equity IS NULL OR debt_to_equity BETWEEN -10 AND 10)
""")

count = spark.sql("SELECT COUNT(*) FROM edgar_kpi_clean").collect()[0][0]
print(f"✅ edgar_kpi_clean created | Rows: {count:,}")